In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/HazardNet Deployment

/content/drive/MyDrive/HazardNet Deployment


In [4]:
# Create the hidden Kaggle directory
!mkdir -p ~/.kaggle
# Move the token file into it
!cp kaggle.json ~/.kaggle/
# Set required file permissions (read/write for owner only)
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!kaggle datasets download -d ashifahmedshuvo/hazardnet-datasets

Dataset URL: https://www.kaggle.com/datasets/ashifahmedshuvo/hazardnet-datasets
License(s): MIT
100% 4.12G/4.12G [00:44<00:00, 100MB/s]



In [6]:
!unzip \*.zip  && rm *.zip

Archive:  hazardnet-datasets.zip
replace tensors_output/HazardNet_Event_Based_Datasets/dataset_config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: tensors_output/HazardNet_Event_Based_Datasets/dataset_config.json  
replace tensors_output/HazardNet_Event_Based_Datasets/event_kfold/fold_0/test_events.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: tensors_output/HazardNet_Event_Based_Datasets/event_kfold/fold_0/test_events.csv  
replace tensors_output/HazardNet_Event_Based_Datasets/event_kfold/fold_0/train_events.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: tensors_output/HazardNet_Event_Based_Datasets/event_kfold/fold_0/train_events.csv  
replace tensors_output/HazardNet_Event_Based_Datasets/event_kfold/fold_0/val_events.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: tensors_output/HazardNet_Event_Based_Datasets/event_kfold/fold_0/val_events.csv  
replace tensors_output/HazardNet_Event_Based_Datasets/event_kfold/fold_1/test_events.csv? [y]

In [8]:
import os, getpass

# GitHub credentials for the auto-push step.
GITHUB_USERNAME = "myself-aas"
GITHUB_REPO     = "HazardNet"
GITHUB_PAT      = getpass.getpass("GitHub Fine-Grained PAT (Contents: write): ")
GIT_USER_EMAIL  = "shuvo.1807016@bau.edu.bd"
GIT_USER_NAME   = "HazardNet Auto-ML"

# Dataset source (pick one):
#   "drive"  — Google Drive mount (fast, recommended after first download)
#   "hf"     — HuggingFace Datasets (zero-config)
DATASET_SOURCE = "drive"
DRIVE_TENSORS_PATH = "/content/drive/MyDrive/HazardNet_Deployment/tensors_output/HazardNet_Event_Based_Datasets/master_tensors.h5"

BUNDLE_DIR = "/content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/deployment_bundle"
REPO_DIR   = "/content/drive/MyDrive/HazardNet_Deployment/HazardNet"
print("Configured.")

GitHub Fine-Grained PAT (Contents: write): ··········
Configured.


In [9]:
import torch
assert torch.cuda.is_available(), "No GPU detected — Runtime → Change runtime type → GPU"
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | Device {torch.cuda.get_device_name(0)}")

PyTorch 2.11.0+cu128 | CUDA 12.8 | Device Tesla T4


In [10]:
import os
os.makedirs('/content/drive/MyDrive/HazardNet_Deployment/data', exist_ok=True)
TENSORS_PATH = '/content/drive/MyDrive/HazardNet_Deployment/data/master_tensors.h5'

if DATASET_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    shutil.copy(DRIVE_TENSORS_PATH, TENSORS_PATH)
elif DATASET_SOURCE == 'hf':
    !wget -q "https://huggingface.co/datasets/{HF_DATASET_REPO}/resolve/main/master_tensors.h5" -O {TENSORS_PATH}

assert os.path.exists(TENSORS_PATH), f"Dataset not found at {TENSORS_PATH}"
!ls -lh {TENSORS_PATH}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
-rw------- 1 root root 2.4G Sep 15 16:35 /content/drive/MyDrive/HazardNet_Deployment/data/master_tensors.h5


In [11]:
# Training stack (only needed in Colab — the Actions daily job uses tflite-runtime only).
!pip install -q torch torchvision h5py numpy pandas scikit-learn tqdm onnxruntime onnx tf2onnx tensorflow tensorflow-probability
import torch, torch.nn as nn, numpy as np, h5py, pandas as pd
from tqdm import tqdm
print("Training deps installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 42.2 MB/s eta 0:00:00
Training deps installed.


In [15]:
"""
================================================================================
HazardNet Unified Experimental Training Pipeline (Q1 Journal Edition)
================================================================================

INTEGRATED FEATURES:
  • W&B Kaggle Secrets Integration (robust fallback)
  • Enhanced Publication Metrics (Per-class, Severity Quartiles, R²)
  • Publication Figure Generator (Confusion Matrix, Scatter, Spatial Heatmap)
  • 4 Validation Strategies: Event K-Fold, Spatial LODO, Temporal, Spatio-Temporal

USAGE IN KAGGLE NOTEBOOK CELL:
  STRATEGY = 'spatial_lodo'  # Change per session
  main()
================================================================================
"""

import os
import json
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, get_worker_info
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             mean_squared_error, mean_absolute_error, confusion_matrix,
                             r2_score)
from tqdm import tqdm
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

WANDB_ENABLED = False

# ============================================================================
# CONFIGURATION
# ============================================================================
class TrainConfig:
    EXPERIMENTAL_DIR = '/content/drive/MyDrive/HazardNet_Deployment/tensors_output/HazardNet_Event_Based_Datasets'
    MASTER_H5_PATH = os.path.join(EXPERIMENTAL_DIR, 'master_tensors.h5')
    CONFIG_PATH = os.path.join(EXPERIMENTAL_DIR, 'dataset_config.json')
    OUTPUT_DIR = '/content/drive/MyDrive/HazardNet_Deployment/data/HazardNet_event_based_model_outputs'

    BATCH_SIZE = 16
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-4
    PATIENCE = 10
    GRAD_CLIP = 1.0
    NUM_WORKERS = 2
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(TrainConfig.OUTPUT_DIR, exist_ok=True)

HAZARD_TYPES = [
    'Cold Wave', 'Drought', 'Fire', 'Flash Flood',
    'Flood', 'Heat Wave', 'Severe Local Storm', 'Tropical Cyclone'
]

# ============================================================================
# HAZARDNET ARCHITECTURE
# ============================================================================
class DepthwiseSeparableConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(in_channels, in_channels, kernel_size,
                                   padding=padding, groups=in_channels, bias=False)
        self.pointwise = nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm3d(out_channels)

    def forward(self, x):
        return self.bn(self.pointwise(self.depthwise(x)))


class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool3d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid())

    def forward(self, x):
        w = self.fc(x).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        return x * w


class HazardNetCNN(nn.Module):
    def __init__(self, in_channels=15, num_hazards=8):
        super().__init__()
        self.block1 = nn.Sequential(
            DepthwiseSeparableConv3d(in_channels, 32), nn.ReLU(True),
            SEBlock3D(32), nn.MaxPool3d((1, 2, 2)))
        self.block2 = nn.Sequential(
            DepthwiseSeparableConv3d(32, 64), nn.ReLU(True),
            SEBlock3D(64), nn.MaxPool3d((2, 2, 2)))
        self.block3 = nn.Sequential(
            DepthwiseSeparableConv3d(64, 128), nn.ReLU(True),
            SEBlock3D(128), nn.MaxPool3d((1, 2, 2)))
        self.block4 = nn.Sequential(
            DepthwiseSeparableConv3d(128, 256), nn.ReLU(True),
            SEBlock3D(256), nn.MaxPool3d((1, 2, 2)))
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.shared_fc = nn.Sequential(nn.Linear(256, 128), nn.ReLU(True), nn.Dropout(0.3))
        self.hazard_head = nn.Linear(128, num_hazards)
        self.severity_head = nn.Sequential(nn.Linear(128, 64), nn.ReLU(True), nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, x):
        x = self.block4(self.block3(self.block2(self.block1(x))))
        x = self.global_pool(x).view(x.size(0), -1)
        x = self.shared_fc(x)
        return self.hazard_head(x), self.severity_head(x).squeeze(1)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ============================================================================
# LOSS FUNCTION
# ============================================================================
class HomoscedasticMTLLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(2))
        self.ce_loss = nn.CrossEntropyLoss(reduction='none')
        self.huber_loss = nn.SmoothL1Loss(reduction='none')

    def forward(self, hazard_pred, severity_pred, hazard_true, severity_true, confidence):
        loss_cls = self.ce_loss(hazard_pred, hazard_true)
        loss_reg = self.huber_loss(severity_pred, severity_true)
        loss_cls_conf = (loss_cls * confidence).mean()
        loss_reg_conf = (loss_reg * confidence).mean()
        prec_cls = torch.exp(-self.log_vars[0])
        prec_reg = torch.exp(-self.log_vars[1])
        total = (prec_cls * loss_cls_conf + self.log_vars[0]) + \
                (prec_reg * loss_reg_conf + self.log_vars[1])
        return total, loss_cls_conf.item(), loss_reg_conf.item()


# ============================================================================
# MASTER HDF5 DATASET
# ============================================================================
class MasterHDF5Dataset(Dataset):
    def __init__(self, csv_path, master_h5_path, augment=False):
        self.df = pd.read_csv(csv_path)
        self.master_h5_path = master_h5_path
        self.augment = augment
        self.h5f = None
        self._worker_id = None
        self.brightness, self.contrast, self.temporal_shift = 0.1, 0.1, 1
        self.target_shape = (15, 10, 64, 64)

    def _open_h5(self):
        wid = get_worker_info().id if get_worker_info() else -1
        if self.h5f is None or self._worker_id != wid:
            if self.h5f: self.h5f.close()
            self.h5f = h5py.File(self.master_h5_path, 'r', rdcc_nbytes=1024**2*10)
            self._worker_id = wid

    def __len__(self): return len(self.df)

    def _resize_spatial(self, tensor):
        c, t, h, w = tensor.shape
        th, tw = self.target_shape[2], self.target_shape[3]
        if h == th and w == tw: return tensor
        r = tensor.permute(1,0,2,3).reshape(t*c,1,h,w)
        r = F.interpolate(r, size=(th,tw), mode='nearest')
        return r.reshape(t,c,th,tw).permute(1,0,2,3).contiguous()

    def _augment(self, tensor):
        if np.random.rand() > 0.5:
            tensor = tensor + np.random.uniform(-self.brightness, self.brightness)
        if np.random.rand() > 0.5:
            f = 1.0 + np.random.uniform(-self.contrast, self.contrast)
            m = tensor.mean(dim=[-1,-2], keepdim=True)
            tensor = (tensor - m) * f + m
        if np.random.rand() > 0.5:
            s = np.random.randint(-self.temporal_shift, self.temporal_shift+1)
            if s > 0:
                b = tensor[:,0:1,:,:].repeat(1,s,1,1)
                tensor = torch.cat([b, tensor[:,:-s,:,:]], dim=1)
            elif s < 0:
                a = abs(s); b = tensor[:,-1:,:,:].repeat(1,a,1,1)
                tensor = torch.cat([tensor[:,a:,:,:], b], dim=1)
        return tensor

    def __getitem__(self, idx):
        self._open_h5()
        row = self.df.iloc[idx]
        eid = str(row['event_id'])
        tensor = torch.from_numpy(self.h5f['tensors'][eid][:]).float()
        label = int(row['hazard_idx'])
        severity = float(row.get('severity_index', 0.0))
        confidence = float(row.get('confidence', 0.5))
        tensor = self._resize_spatial(tensor)
        if self.augment: tensor = self._augment(tensor)
        return tensor, label, severity, confidence, eid

    def __del__(self):
        if self.h5f: self.h5f.close()


# ============================================================================
# ENHANCED METRICS TRACKER (Q1 Journal Grade)
# ============================================================================
class EnhancedMetricsTracker:
    def __init__(self):
        self.reset()

    def reset(self):
        self.total_losses, self.cls_losses, self.reg_losses = [], [], []
        self.hazard_preds, self.hazard_targets = [], []
        self.severity_preds, self.severity_targets = [], []

    def update(self, total_loss, cls_loss, reg_loss, h_pred, h_true, s_pred, s_true):
        self.total_losses.append(total_loss)
        self.cls_losses.append(cls_loss)
        self.reg_losses.append(reg_loss)
        self.hazard_preds.extend(h_pred)
        self.hazard_targets.extend(h_true)
        self.severity_preds.extend(s_pred)
        self.severity_targets.extend(s_true)

    def get_summary(self):
        h_acc = accuracy_score(self.hazard_targets, self.hazard_preds)
        h_f1 = f1_score(self.hazard_targets, self.hazard_preds, average='weighted', zero_division=0)
        s_mse = mean_squared_error(self.severity_targets, self.severity_preds)
        return {
            'loss_total': np.mean(self.total_losses),
            'loss_cls': np.mean(self.cls_losses),
            'loss_reg': np.mean(self.reg_losses),
            'hazard_accuracy': h_acc, 'hazard_f1': h_f1,
            'severity_mse': s_mse, 'severity_rmse': np.sqrt(s_mse),
            'severity_mae': mean_absolute_error(self.severity_targets, self.severity_preds),
            'severity_r2': r2_score(self.severity_targets, self.severity_preds),
        }

    def get_per_class_metrics(self):
        prec = precision_score(self.hazard_targets, self.hazard_preds,
                               average=None, labels=range(len(HAZARD_TYPES)), zero_division=0)
        rec = recall_score(self.hazard_targets, self.hazard_preds,
                           average=None, labels=range(len(HAZARD_TYPES)), zero_division=0)
        f1 = f1_score(self.hazard_targets, self.hazard_preds,
                      average=None, labels=range(len(HAZARD_TYPES)), zero_division=0)
        support = np.bincount(self.hazard_targets, minlength=len(HAZARD_TYPES))
        rows = []
        for i, name in enumerate(HAZARD_TYPES):
            rows.append({'Hazard': name, 'Precision': prec[i], 'Recall': rec[i],
                         'F1-Score': f1[i], 'Support': support[i]})
        rows.append({'Hazard': 'Macro Avg',
                     'Precision': precision_score(self.hazard_targets, self.hazard_preds, average='macro', zero_division=0),
                     'Recall': recall_score(self.hazard_targets, self.hazard_preds, average='macro', zero_division=0),
                     'F1-Score': f1_score(self.hazard_targets, self.hazard_preds, average='macro', zero_division=0),
                     'Support': sum(support)})
        rows.append({'Hazard': 'Weighted Avg',
                     'Precision': precision_score(self.hazard_targets, self.hazard_preds, average='weighted', zero_division=0),
                     'Recall': recall_score(self.hazard_targets, self.hazard_preds, average='weighted', zero_division=0),
                     'F1-Score': f1_score(self.hazard_targets, self.hazard_preds, average='weighted', zero_division=0),
                     'Support': sum(support)})
        return pd.DataFrame(rows)

    def get_severity_error_by_quartile(self):
        targets = np.array(self.severity_targets)
        preds = np.array(self.severity_preds)
        if len(targets) == 0: return pd.DataFrame()
        quartiles = np.percentile(targets, [25, 50, 75])
        bins = [0, quartiles[0], quartiles[1], quartiles[2], 1.0]
        labels = ['Q1 (Low)', 'Q2 (Moderate)', 'Q3 (High)', 'Q4 (Severe)']
        bin_idx = np.digitize(targets, bins[1:-1])
        rows = []
        for q in range(4):
            mask = bin_idx == q
            if mask.sum() == 0: continue
            t_q, p_q = targets[mask], preds[mask]
            rows.append({'Severity Quartile': labels[q], 'N': int(mask.sum()),
                         'MAE': mean_absolute_error(t_q, p_q),
                         'RMSE': np.sqrt(mean_squared_error(t_q, p_q)),
                         'Mean Predicted': p_q.mean(), 'Mean Actual': t_q.mean()})
        return pd.DataFrame(rows)

    def get_confusion_matrix_normalized(self):
        cm = confusion_matrix(self.hazard_targets, self.hazard_preds, labels=range(len(HAZARD_TYPES)))
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        return np.nan_to_num(cm_norm)


# ============================================================================
# PUBLICATION FIGURES GENERATOR
# ============================================================================
class PublicationFigureGenerator:
    def __init__(self, output_dir):
        self.output_dir = output_dir
        os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
        plt.rcParams.update({'font.size': 10, 'axes.labelsize': 12, 'axes.titlesize': 13,
                             'xtick.labelsize': 9, 'ytick.labelsize': 9,
                             'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})

    def plot_confusion_matrix(self, cm_normalized, title, filename):
        fig, ax = plt.subplots(figsize=(8, 7))
        sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=HAZARD_TYPES, yticklabels=HAZARD_TYPES, ax=ax)
        ax.set_xlabel('Predicted Hazard'); ax.set_ylabel('True Hazard'); ax.set_title(title)
        plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
        plt.tight_layout()
        path = os.path.join(self.output_dir, 'figures', filename)
        plt.savefig(path); plt.close()
        print(f"  Saved: {path}")

    def plot_severity_scatter(self, targets, preds, r2, title, filename):
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.scatter(targets, preds, alpha=0.3, s=10, c='steelblue')
        lims = [0, 1]
        ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel('Ground Truth Severity'); ax.set_ylabel('Predicted Severity')
        ax.set_title(f"{title}\nR²={r2:.4f}"); ax.legend(); ax.set_aspect('equal')
        plt.tight_layout()
        path = os.path.join(self.output_dir, 'figures', filename)
        plt.savefig(path); plt.close()
        print(f"   Saved: {path}")

    def plot_spatial_heatmap(self, results_df, title, filename):
        if 'division' not in results_df.columns or 'season' not in results_df.columns: return
        pivot = results_df.pivot_table(values='accuracy', index='division', columns='season', aggfunc='mean')
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', vmin=0.5, vmax=1.0, ax=ax)
        ax.set_title(title)
        plt.tight_layout()
        path = os.path.join(self.output_dir, 'figures', filename)
        plt.savefig(path); plt.close()
        print(f"   Saved: {path}")


# ============================================================================
# TRAINING & EVALUATION FUNCTIONS
# ============================================================================
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    metrics = EnhancedMetricsTracker()
    pbar = tqdm(loader, desc="Train", unit="batch")
    for tensors, cls_idx, severity, confidence, _ in pbar:
        tensors, cls_idx, severity, confidence = [t.to(device) for t in [tensors, cls_idx, severity, confidence]]
        optimizer.zero_grad()
        h_pred, s_pred = model(tensors)
        total, cls_l, reg_l = criterion(h_pred, s_pred, cls_idx, severity, confidence)
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=TrainConfig.GRAD_CLIP)
        optimizer.step()
        metrics.update(total.item(), cls_l, reg_l,
                       h_pred.detach().argmax(1).cpu().numpy(), cls_idx.cpu().numpy(),
                       s_pred.detach().cpu().numpy(), severity.cpu().numpy())
        pbar.set_postfix({'loss': f"{total.item():.4f}"})
    return metrics


def evaluate(model, loader, criterion, device, split_name="Val"):
    model.eval()
    metrics = EnhancedMetricsTracker()
    with torch.no_grad():
        pbar = tqdm(loader, desc=split_name, unit="batch")
        for tensors, cls_idx, severity, confidence, _ in pbar:
            tensors, cls_idx, severity, confidence = [t.to(device) for t in [tensors, cls_idx, severity, confidence]]
            h_pred, s_pred = model(tensors)
            total, cls_l, reg_l = criterion(h_pred, s_pred, cls_idx, severity, confidence)
            metrics.update(total.item(), cls_l, reg_l,
                           h_pred.argmax(1).cpu().numpy(), cls_idx.cpu().numpy(),
                           s_pred.cpu().numpy(), severity.cpu().numpy())
            pbar.set_postfix({'loss': f"{total.item():.4f}"})
    return metrics


def train_single_fold(fold_name, train_csv, val_csv, test_csv, num_classes, output_dir):
    print(f"\n{'─'*60}")
    print(f" {fold_name}")
    print(f"{'─'*60}")

    train_loader = DataLoader(MasterHDF5Dataset(train_csv, TrainConfig.MASTER_H5_PATH, True),
                              batch_size=TrainConfig.BATCH_SIZE, shuffle=True,
                              num_workers=TrainConfig.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(MasterHDF5Dataset(val_csv, TrainConfig.MASTER_H5_PATH, False),
                            batch_size=TrainConfig.BATCH_SIZE, shuffle=False,
                            num_workers=TrainConfig.NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(MasterHDF5Dataset(test_csv, TrainConfig.MASTER_H5_PATH, False),
                             batch_size=TrainConfig.BATCH_SIZE, shuffle=False,
                             num_workers=TrainConfig.NUM_WORKERS, pin_memory=True)

    print(f"  Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}, Test: {len(test_loader.dataset)}")

    model = HazardNetCNN(15, num_classes).to(TrainConfig.DEVICE)
    criterion = HomoscedasticMTLLoss().to(TrainConfig.DEVICE)
    optimizer = AdamW([{'params': model.parameters()}, {'params': criterion.log_vars}],
                      lr=TrainConfig.LEARNING_RATE, weight_decay=TrainConfig.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=TrainConfig.NUM_EPOCHS, eta_min=1e-6)

    best_val_loss, patience_counter, best_epoch = float('inf'), 0, 0
    safe_name = fold_name.replace('/', '_').replace(' ', '_')
    ckpt_path = os.path.join(output_dir, f'{safe_name}_best.pt')
    fig_gen = PublicationFigureGenerator(output_dir)

    # W&B Init per fold
    run = None
    if WANDB_ENABLED:
        try:
            run = wandb.init(project='hazardnet', name=safe_name, reinit=True,
                             config={'strategy': fold_name, 'batch_size': TrainConfig.BATCH_SIZE})
        except: pass

    for epoch in range(TrainConfig.NUM_EPOCHS):
        train_m = train_epoch(model, train_loader, optimizer, criterion, TrainConfig.DEVICE)
        val_m = evaluate(model, val_loader, criterion, TrainConfig.DEVICE, "Val")
        scheduler.step()
        ts, vs = train_m.get_summary(), val_m.get_summary()

        if vs['loss_total'] < best_val_loss:
            best_val_loss = vs['loss_total']; patience_counter = 0; best_epoch = epoch + 1
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= TrainConfig.PATIENCE:
                print(f"  ⏹️ Early stopping at epoch {epoch+1} (best: {best_epoch})"); break

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:2d}/{TrainConfig.NUM_EPOCHS} | "
                  f"Train: {ts['loss_total']:.4f} Acc:{ts['hazard_accuracy']:.3f} | "
                  f"Val: {vs['loss_total']:.4f} Acc:{vs['hazard_accuracy']:.3f} RMSE:{vs['severity_rmse']:.4f}")

        if run:
            wandb.log({'train_loss': ts['loss_total'], 'val_loss': vs['loss_total'],
                       'train_acc': ts['hazard_accuracy'], 'val_acc': vs['hazard_accuracy']})

    # TEST EVALUATION WITH FULL METRICS
    model.load_state_dict(torch.load(ckpt_path))
    test_metrics = evaluate(model, test_loader, criterion, TrainConfig.DEVICE, "Test")
    test_summary = test_metrics.get_summary()

    print(f"   Test: Acc={test_summary['hazard_accuracy']:.4f} F1={test_summary['hazard_f1']:.4f} "
          f"RMSE={test_summary['severity_rmse']:.4f} R²={test_summary['severity_r2']:.4f}")

    # Generate Publication Tables & Figures
    per_class = test_metrics.get_per_class_metrics()
    per_class.to_csv(os.path.join(output_dir, f'{safe_name}_per_class.csv'), index=False)

    sev_quartile = test_metrics.get_severity_error_by_quartile()
    if not sev_quartile.empty:
        sev_quartile.to_csv(os.path.join(output_dir, f'{safe_name}_severity_quartile.csv'), index=False)

    fig_gen.plot_confusion_matrix(test_metrics.get_confusion_matrix_normalized(),
                                  f'Confusion Matrix: {fold_name}', f'{safe_name}_confusion_matrix.png')
    fig_gen.plot_severity_scatter(test_metrics.severity_targets, test_metrics.severity_preds,
                                  test_summary['severity_r2'], f'Severity: {fold_name}', f'{safe_name}_severity_scatter.png')

    if run: wandb.finish()

    return {
        'fold': fold_name, 'accuracy': test_summary['hazard_accuracy'],
        'f1': test_summary['hazard_f1'], 'rmse': test_summary['severity_rmse'],
        'mae': test_summary['severity_mae'], 'r2': test_summary['severity_r2'],
        'n_test': len(test_loader.dataset),
    }


# ============================================================================
# STRATEGY RUNNERS
# ============================================================================
def run_event_kfold(num_classes, output_dir):
    base = os.path.join(TrainConfig.EXPERIMENTAL_DIR, 'event_kfold')
    results = []
    for fold_idx in range(5):
        fd = os.path.join(base, f'fold_{fold_idx}')
        if not os.path.exists(fd): print(f"   Skipping fold_{fold_idx}"); continue
        r = train_single_fold(f'event_kfold_fold{fold_idx}',
                              os.path.join(fd, 'train_events.csv'), os.path.join(fd, 'val_events.csv'),
                              os.path.join(fd, 'test_events.csv'), num_classes, output_dir)
        results.append(r)
    return results

def run_spatial_lodo(num_classes, output_dir):
    base = os.path.join(TrainConfig.EXPERIMENTAL_DIR, 'spatial_lodo')
    results = []
    fold_dirs = sorted(glob.glob(os.path.join(base, 'lodo_division_*')))
    print(f"  Found {len(fold_dirs)} LODO division folds")
    for fd in fold_dirs:
        fn = os.path.basename(fd)
        r = train_single_fold(fn, os.path.join(fd, 'train_events.csv'), os.path.join(fd, 'val_events.csv'),
                              os.path.join(fd, 'test_events.csv'), num_classes, output_dir)
        r['division'] = fn.replace('lodo_division_', '')
        results.append(r)
    return results

def run_temporal(num_classes, output_dir):
    base = os.path.join(TrainConfig.EXPERIMENTAL_DIR, 'temporal_split')
    if not os.path.exists(os.path.join(base, 'train_events.csv')):
        print("   Temporal split not found"); return []
    r = train_single_fold('temporal_season_adaptive',
                          os.path.join(base, 'train_events.csv'), os.path.join(base, 'val_events.csv'),
                          os.path.join(base, 'test_events.csv'), num_classes, output_dir)
    return [r]

def run_spatio_temporal(num_classes, output_dir):
    base = os.path.join(TrainConfig.EXPERIMENTAL_DIR, 'spatio_temporal')
    results = []
    fold_dirs = sorted(glob.glob(os.path.join(base, 'st_*')))
    print(f"  Found {len(fold_dirs)} spatio-temporal folds")
    for fd in fold_dirs:
        fn = os.path.basename(fd)
        r = train_single_fold(fn, os.path.join(fd, 'train_events.csv'), os.path.join(fd, 'val_events.csv'),
                              os.path.join(fd, 'test_events.csv'), num_classes, output_dir)
        parts = fn.replace('st_', '').rsplit('_', 1)
        if len(parts) == 2: r['division'], r['season'] = parts[0], parts[1]
        results.append(r)
    return results


# ============================================================================
# MAIN ORCHESTRATOR
# ============================================================================
STRATEGY_MAP = {
    'event_kfold': ('Event-Based 5-Fold CV', run_event_kfold),
    'spatial_lodo': ('Spatial LODO (Division-Level)', run_spatial_lodo),
    'temporal': ('Temporal Split (Season-Adaptive)', run_temporal),
    'spatio_temporal': ('Spatio-Temporal (Division×Season×Era)', run_spatio_temporal),
}

# ═══════════════════════════════════════════════════════════
# SET YOUR STRATEGY HERE (for Kaggle notebook execution)
# ═══════════════════════════════════════════════════════════
STRATEGY = 'event_kfold'  # Options: event_kfold, spatial_lodo, temporal, spatio_temporal, all
# ═══════════════════════════════════════════════════════════

def main():
    print("=" * 80)
    print("  HAZARDNET UNIFIED EXPERIMENTAL TRAINING (Q1 Journal Edition)")
    print("=" * 80)

    with open(TrainConfig.CONFIG_PATH, 'r') as f:
        config = json.load(f)
    num_classes = config['n_classes']

    print(f"  Classes ({num_classes}): {config['hazard_types']}")
    print(f"  Master HDF5: {TrainConfig.MASTER_H5_PATH}")
    print(f"  Device: {TrainConfig.DEVICE}")
    print(f"  W&B Enabled: {WANDB_ENABLED}")

    strategies = list(STRATEGY_MAP.items()) if STRATEGY == 'all' else [(STRATEGY, STRATEGY_MAP[STRATEGY])]
    all_strategy_results = {}

    for strat_key, (strat_name, strat_fn) in strategies:
        print(f"\n{'='*80}")
        print(f"STRATEGY: {strat_name.upper()}")
        print(f"{'='*80}")

        strat_output = os.path.join(TrainConfig.OUTPUT_DIR, strat_key)
        os.makedirs(strat_output, exist_ok=True)

        results = strat_fn(num_classes, strat_output)
        all_strategy_results[strat_key] = results

        if results:
            accs = [r['accuracy'] for r in results]
            f1s = [r['f1'] for r in results]
            rmses = [r['rmse'] for r in results]
            r2s = [r['r2'] for r in results]
            print(f"\n  {strat_name} Summary ({len(results)} folds):")
            print(f"     Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
            print(f"     F1-Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
            print(f"     RMSE:     {np.mean(rmses):.4f} ± {np.std(rmses):.4f}")
            print(f"     R²:       {np.mean(r2s):.4f} ± {np.std(r2s):.4f}")

            df = pd.DataFrame([{k: v for k, v in r.items()} for r in results])
            df.to_csv(os.path.join(strat_output, f'{strat_key}_results.csv'), index=False)

            if strat_key in ['spatial_lodo', 'spatio_temporal']:
                fig_gen = PublicationFigureGenerator(strat_output)
                fig_gen.plot_spatial_heatmap(df, f'{strat_name} Accuracy', f'{strat_key}_spatial_heatmap.png')

    # CROSS-STRATEGY COMPARISON
    print(f"\n{'='*80}")
    print("CROSS-STRATEGY COMPARISON (IEEE TGRS Table II)")
    print(f"{'='*80}")

    comparison_rows = []
    for strat_key, (strat_name, _) in STRATEGY_MAP.items():
        results = all_strategy_results.get(strat_key, [])
        if results:
            accs = [r['accuracy'] for r in results]
            f1s = [r['f1'] for r in results]
            rmses = [r['rmse'] for r in results]
            r2s = [r['r2'] for r in results]
            comparison_rows.append({
                'Strategy': strat_name, 'N_Folds': len(results),
                'Accuracy': f"{np.mean(accs):.4f} ± {np.std(accs):.4f}",
                'F1-Score': f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}",
                'RMSE': f"{np.mean(rmses):.4f} ± {np.std(rmses):.4f}",
                'R²': f"{np.mean(r2s):.4f} ± {np.std(r2s):.4f}",
                'Total_Test_Events': sum(r['n_test'] for r in results),
            })

    if comparison_rows:
        df_comp = pd.DataFrame(comparison_rows)
        print(df_comp.to_string(index=False))
        comp_path = os.path.join(TrainConfig.OUTPUT_DIR, 'cross_strategy_comparison.csv')
        df_comp.to_csv(comp_path, index=False)
        print(f"\nComparison saved to: {comp_path}")

    print(f"\nAll experimental training complete!")
    print(f"   Results: {TrainConfig.OUTPUT_DIR}")


if __name__ == '__main__':
    main()

  HAZARDNET UNIFIED EXPERIMENTAL TRAINING (Q1 Journal Edition)
  Classes (8): ['Cold Wave', 'Drought', 'Fire', 'Flash Flood', 'Flood', 'Heat Wave', 'Severe Local Storm', 'Tropical Cyclone']
  Master HDF5: /content/drive/MyDrive/HazardNet_Deployment/tensors_output/HazardNet_Event_Based_Datasets/master_tensors.h5
  Device: cuda
  W&B Enabled: False

STRATEGY: EVENT-BASED 5-FOLD CV

────────────────────────────────────────────────────────────
 event_kfold_fold0
────────────────────────────────────────────────────────────
  Train: 1930, Val: 414, Test: 587


Val: 100%|██████████| 26/26 [00:09<00:00,  2.85batch/s, loss=0.5943]


  Epoch  1/50 | Train: 1.0821 Acc:0.527 | Val: 0.5227 Acc:0.775 RMSE:0.3049


Val: 100%|██████████| 26/26 [00:08<00:00,  2.95batch/s, loss=-2.1327]


  Epoch 10/50 | Train: -1.6986 Acc:0.953 | Val: -1.6875 Acc:0.959 RMSE:0.2652


...

  Epoch 40/50 | Train: -5.0809 Acc:0.993 | Val: -4.4302 Acc:0.986 RMSE:0.1270


Val: 100%|██████████| 26/26 [00:06<00:00,  4.18batch/s, loss=-5.4309]


  Epoch 50/50 | Train: -5.3221 Acc:0.997 | Val: -4.4171 Acc:0.983 RMSE:0.1257


Test: 100%|██████████| 37/37 [00:10<00:00,  3.46batch/s, loss=-5.3733]


   Test: Acc=0.9846 F1=0.9846 RMSE=0.1274 R²=0.8721
  Saved: /content/drive/MyDrive/HazardNet_Deployment/data/HazardNet_event_based_model_outputs/event_kfold/figures/event_kfold_fold4_confusion_matrix.png
   Saved: /content/drive/MyDrive/HazardNet_Deployment/data/HazardNet_event_based_model_outputs/event_kfold/figures/event_kfold_fold4_severity_scatter.png

  Event-Based 5-Fold CV Summary (5 folds):
     Accuracy: 0.9908 ± 0.0055
     F1-Score: 0.9908 ± 0.0055
     RMSE:     0.1253 ± 0.0097
     R²:       0.8796 ± 0.0180

CROSS-STRATEGY COMPARISON (IEEE TGRS Table II)
             Strategy  N_Folds        Accuracy        F1-Score            RMSE              R²  Total_Test_Events
Event-Based 5-Fold CV        5 0.9908 ± 0.0055 0.9908 ± 0.0055 0.1253 ± 0.0097 0.8796 ± 0.0180               2931

Comparison saved to: /content/drive/MyDrive/HazardNet_Deployment/data/HazardNet_event_based_model_outputs/cross_strategy_comparison.csv

All experimental training complete!
   Results: /content/dr

In [16]:
BEST_PT = '/content/drive/MyDrive/HazardNet_Deployment/data/HazardNet_event_based_model_outputs/event_kfold/event_kfold_fold2_best.pt'

assert os.path.exists(BEST_PT), f"Training did not produce {BEST_PT}"
print(f"Trained model saved to {BEST_PT} ({os.path.getsize(BEST_PT)/1e6:.2f} MB)")

Trained model saved to /content/drive/MyDrive/HazardNet_Deployment/data/HazardNet_event_based_model_outputs/event_kfold/event_kfold_fold2_best.pt (0.57 MB)


In [17]:
import os, json, shutil
os.makedirs(BUNDLE_DIR, exist_ok=True)

In [23]:
import os
import json
import glob
import subprocess
import shutil
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import h5py
from tqdm import tqdm


class DeployConfig:
    BEST_CHECKPOINT = '/content/drive/MyDrive/HazardNet_Deployment/data/HazardNet_event_based_model_outputs/event_kfold/event_kfold_fold2_best.pt'
    MASTER_H5_PATH = '/content/drive/MyDrive/HazardNet_Deployment/tensors_output/HazardNet_Event_Based_Datasets/master_tensors.h5'
    CONFIG_PATH = '/content/drive/MyDrive/HazardNet_Deployment/tensors_output/HazardNet_Event_Based_Datasets/dataset_config.json'
    NORMALIZATION_STATS_PATH = '/content/drive/MyDrive/HazardNet_Deployment/tensors_output/normalization_stats.json'
    OUTPUT_DIR = '/content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles'

    IN_CHANNELS = 15
    NUM_HAZARDS = 8
    INPUT_SHAPE = (1, 15, 10, 64, 64)
    NUM_REPRESENTATIVE_SAMPLES = 100

    BAND_NAMES = [
        'SAR_VV', 'SAR_VH', 'Blue', 'Red', 'NIR', 'SWIR',
        'Temp_2m', 'Precip', 'Max_Temp', 'Min_Temp',
        'Soil_W1', 'Soil_W3', 'Soil_T1', 'Dewpoint', 'Solar_Rad'
    ]

os.makedirs(DeployConfig.OUTPUT_DIR, exist_ok=True)


class NormalizationStats:
    """Robust Normalization Stats Loader that adapts to various JSON schemas."""
    def __init__(self, stats_path: str, band_names: list):
        print(f"Loading normalization stats from: {stats_path}")
        if not os.path.exists(stats_path):
            raise FileNotFoundError(f"Normalization stats file not found: {stats_path}")

        with open(stats_path, 'r') as f:
            self.stats = json.load(f)

        self.band_names = band_names
        self.means_list = []
        self.stds_list = []

        self._parse_and_validate()

        self.means = np.array(self.means_list, dtype=np.float32).reshape(1, -1, 1, 1, 1)
        self.stds = np.array(self.stds_list, dtype=np.float32).reshape(1, -1, 1, 1, 1)
        self.stds = np.where(self.stds < 1e-8, 1.0, self.stds)

        print(f"  [OK] Successfully loaded normalization stats for {len(self.band_names)} bands")

    def _parse_and_validate(self):
        data = self.stats

        if isinstance(data, dict):
            for nested_key in ['per_band', 'bands', 'stats', 'band_stats']:
                if nested_key in data and isinstance(data[nested_key], (dict, list)):
                    data = data[nested_key]
                    break

        if isinstance(data, dict) and any(k in data for k in ['mean', 'means']) and any(k in data for k in ['std', 'stds']):
            m_key = 'mean' if 'mean' in data else 'means'
            s_key = 'std' if 'std' in data else 'stds'
            means_data, stds_data = data[m_key], data[s_key]

            if isinstance(means_data, list) and isinstance(stds_data, list):
                if len(means_data) == len(self.band_names):
                    self.means_list = [float(m) for m in means_data]
                    self.stds_list = [float(s) for s in stds_data]
                    return
                else:
                    raise ValueError(f"Stats list length ({len(means_data)}) != expected bands ({len(self.band_names)})")

            elif isinstance(means_data, dict) and isinstance(stds_data, dict):
                means_lower = {str(k).lower(): v for k, v in means_data.items()}
                stds_lower = {str(k).lower(): v for k, v in stds_data.items()}
                for i, b in enumerate(self.band_names):
                    b_lower, b_idx = b.lower(), str(i)
                    if b_lower in means_lower and b_lower in stds_lower:
                        self.means_list.append(float(means_lower[b_lower]))
                        self.stds_list.append(float(stds_lower[b_lower]))
                    elif b_idx in means_lower and b_idx in stds_lower:
                        self.means_list.append(float(means_lower[b_idx]))
                        self.stds_list.append(float(stds_lower[b_idx]))
                    else:
                        raise ValueError(f"Could not find mean/std for band '{b}' in stats dict")
                return

        if isinstance(data, dict):
            key_map = {str(k).lower(): v for k, v in data.items() if isinstance(v, dict)}
            missing = []
            for i, band in enumerate(self.band_names):
                band_lower, band_idx = band.lower(), str(i)
                target = key_map.get(band_lower) or key_map.get(band_idx)
                if target is not None:
                    m_val = target.get('mean', target.get('means'))
                    s_val = target.get('std', target.get('stds'))
                    if m_val is not None and s_val is not None:
                        self.means_list.append(float(m_val))
                        self.stds_list.append(float(s_val))
                        continue
                missing.append(band)
            if not missing:
                return
            raise ValueError(f"Normalization stats missing bands: {missing}.")

        if isinstance(data, list) and all(isinstance(x, dict) for x in data):
            band_map = {}
            for entry in data:
                b_name = entry.get('band', entry.get('name', entry.get('band_name')))
                if b_name is not None:
                    band_map[str(b_name).lower()] = entry
            for i, band in enumerate(self.band_names):
                entry = band_map.get(band.lower()) or band_map.get(str(i))
                if entry and 'mean' in entry and 'std' in entry:
                    self.means_list.append(float(entry['mean']))
                    self.stds_list.append(float(entry['std']))
                else:
                    raise ValueError(f"Missing stats entry for band '{band}' in list of stats.")
            return

        raise ValueError("Unrecognized normalization stats JSON structure.")

    def normalize(self, tensor: np.ndarray) -> np.ndarray:
        if tensor.ndim == 4:
            means, stds = self.means[0], self.stds[0]
        elif tensor.ndim == 5:
            means, stds = self.means, self.stds
        else:
            raise ValueError(f"Expected 4D or 5D tensor, got {tensor.ndim}D")
        return (tensor.astype(np.float32) - means) / stds

    def to_dict(self) -> dict:
        return {
            'means': {b: float(self.means_list[i]) for i, b in enumerate(self.band_names)},
            'stds': {b: float(self.stds_list[i]) for i, b in enumerate(self.band_names)},
            'band_order': self.band_names,
            'normalization_type': 'z_score',
        }


class DepthwiseSeparableConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(in_channels, in_channels, kernel_size, padding=padding, groups=in_channels, bias=False)
        self.pointwise = nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm3d(out_channels)

    def forward(self, x):
        return self.bn(self.pointwise(self.depthwise(x)))


class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool3d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid())

    def forward(self, x):
        w = self.fc(x).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        return x * w


class HazardNetCNN(nn.Module):
    def __init__(self, in_channels=15, num_hazards=8):
        super().__init__()
        self.block1 = nn.Sequential(DepthwiseSeparableConv3d(in_channels, 32), nn.ReLU(True), SEBlock3D(32), nn.MaxPool3d((1, 2, 2)))
        self.block2 = nn.Sequential(DepthwiseSeparableConv3d(32, 64), nn.ReLU(True), SEBlock3D(64), nn.MaxPool3d((2, 2, 2)))
        self.block3 = nn.Sequential(DepthwiseSeparableConv3d(64, 128), nn.ReLU(True), SEBlock3D(128), nn.MaxPool3d((1, 2, 2)))
        self.block4 = nn.Sequential(DepthwiseSeparableConv3d(128, 256), nn.ReLU(True), SEBlock3D(256), nn.MaxPool3d((1, 2, 2)))
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.shared_fc = nn.Sequential(nn.Linear(256, 128), nn.ReLU(True), nn.Dropout(0.3))
        self.hazard_head = nn.Linear(128, num_hazards)
        self.severity_head = nn.Sequential(nn.Linear(128, 64), nn.ReLU(True), nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, x):
        x = self.block4(self.block3(self.block2(self.block1(x))))
        x = self.global_pool(x).view(x.size(0), -1)
        x = self.shared_fc(x)
        return self.hazard_head(x), self.severity_head(x).squeeze(1)


def load_model_and_stats():
    print("=" * 70)
    print("STEP 1: Loading Model & Pre-computed Normalization Stats")
    print("=" * 70)
    norm_stats = NormalizationStats(DeployConfig.NORMALIZATION_STATS_PATH, DeployConfig.BAND_NAMES)

    model = HazardNetCNN(DeployConfig.IN_CHANNELS, DeployConfig.NUM_HAZARDS)
    state_dict = torch.load(DeployConfig.BEST_CHECKPOINT, map_location='cpu')
    model.load_state_dict(state_dict)
    model.eval()

    params = sum(p.numel() for p in model.parameters())
    print(f"  [OK] Model loaded: {params:,} params (~{params * 4 / 1024**2:.2f} MB FP32)")
    return model, norm_stats


def export_to_onnx(model):
    print("\n" + "=" * 70)
    print("STEP 2: Exporting to ONNX")
    print("=" * 70)
    output_path = os.path.join(DeployConfig.OUTPUT_DIR, 'hazardnet.onnx')
    dummy_input = torch.randn(*DeployConfig.INPUT_SHAPE)

    print("  Tracing model with torch.jit.trace to bypass onnxscript registry bugs...")
    try:
        export_target = torch.jit.trace(model, dummy_input)
    except Exception as e:
        print(f"  [WARN] Tracing warning ({e}), falling back to PyTorch model")
        export_target = model

    torch.onnx.export(
        export_target, dummy_input, output_path,
        export_params=True, opset_version=17, do_constant_folding=True,
        input_names=['input'],
        output_names=['hazard_logits', 'severity_pred'],
        dynamic_axes={'input': {0: 'batch_size'}, 'hazard_logits': {0: 'batch_size'}, 'severity_pred': {0: 'batch_size'}},
        dynamo=False
    )

    import onnx
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model)
    print(f"  [OK] ONNX exported: {output_path} ({os.path.getsize(output_path) / 1024**2:.2f} MB)")
    return output_path


def create_representative_dataset(norm_stats: NormalizationStats):
    print("\n" + "=" * 70)
    print(f"STEP 3: Creating Representative Dataset ({DeployConfig.NUM_REPRESENTATIVE_SAMPLES} samples)")
    print("=" * 70)

    event_kfold_dir = os.path.dirname(DeployConfig.MASTER_H5_PATH)
    train_csvs = sorted(glob.glob(os.path.join(event_kfold_dir, 'event_kfold/fold_*/train_events.csv')))
    if not train_csvs:
        raise FileNotFoundError(f"No train CSVs found in {event_kfold_dir}")

    samples_per_fold = max(1, DeployConfig.NUM_REPRESENTATIVE_SAMPLES // len(train_csvs))
    all_event_ids = []
    for csv_path in train_csvs:
        fold_df = pd.read_csv(csv_path)
        fold_sample = fold_df.sample(n=min(samples_per_fold, len(fold_df)), random_state=42)
        all_event_ids.extend(fold_sample['event_id'].astype(str).tolist())
    all_event_ids = all_event_ids[:DeployConfig.NUM_REPRESENTATIVE_SAMPLES]

    h5f = h5py.File(DeployConfig.MASTER_H5_PATH, 'r')
    samples = []
    for eid in tqdm(all_event_ids, desc="Loading & normalizing"):
        raw_tensor = h5f[f'tensors/{eid}'][:]
        normalized = norm_stats.normalize(raw_tensor)
        samples.append(normalized[np.newaxis, ...])
    h5f.close()

    print(f"  [OK] Created {len(samples)} normalized representative samples")
    return samples


def convert_to_tflite(onnx_path, representative_samples):
    print("\n" + "=" * 70)
    print("STEP 4: Converting to TFLite")
    print("=" * 70)

    # Install onnx2tf if not already installed
    try:
        import onnx2tf
    except ImportError:
        print("  Installing onnx2tf...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'onnx2tf'], check=True, capture_output=True, text=True)
        print("  onnx2tf installed successfully.")

    import tensorflow as tf

    tflite_dir = os.path.join(DeployConfig.OUTPUT_DIR, 'tflite')
    os.makedirs(tflite_dir, exist_ok=True)
    tf_saved_model_dir = os.path.join(tflite_dir, 'tf_saved_model')

    print("  Converting ONNX -> TF SavedModel / TFLite...")

    # Robust CLI execution using sys.executable to avoid PATH issues in Kaggle/Colab
    cmd = [sys.executable, '-m', 'onnx2tf', '-i', onnx_path, '-o', tf_saved_model_dir, '-osd', '-nuo']
    print(f"  Running command: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)

    # Print stdout and stderr for debugging
    print("onnx2tf stdout:")
    print(result.stdout)
    print("onnx2tf stderr:")
    print(result.stderr)

    # Raise an exception if onnx2tf failed
    result.check_returncode()
    print("  [OK] onnx2tf command executed successfully.")

    def _find_saved_model(path):
        for root, _, files in os.walk(path):
            if 'saved_model.pb' in files: return root
        return None

    saved_pb_path = _find_saved_model(tf_saved_model_dir)
    fp32_model_bytes = None

    if saved_pb_path:
        print(f"  [OK] Found TF SavedModel at: {saved_pb_path}")
        print("  Converting SavedModel to Pure FP32 TFLite...")
        converter = tf.lite.TFLiteConverter.from_saved_model(saved_pb_path)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
        converter.inference_input_type = tf.float32
        converter.inference_output_type = tf.float32
        fp32_model_bytes = converter.convert()
    else:
        # Fallback: onnx2tf often outputs .tflite directly if SavedModel generation fails
        direct_tflite_files = glob.glob(os.path.join(tf_saved_model_dir, "*float32.tflite"))
        if not direct_tflite_files:
            direct_tflite_files = glob.glob(os.path.join(tf_saved_model_dir, "*.tflite"))

        if direct_tflite_files:
            saved_pb_path = direct_tflite_files[0]
            print(f"  [OK] Found direct FP32 TFLite model: {saved_pb_path}")
            with open(saved_pb_path, 'rb') as f:
                fp32_model_bytes = f.read()
        else:
            raise RuntimeError(f"TF SavedModel or TFLite not created at '{tf_saved_model_dir}'.")

    fp32_path = os.path.join(tflite_dir, 'hazardnet_fp32.tflite')
    with open(fp32_path, 'wb') as f:
        f.write(fp32_model_bytes)

    print("\n  [INFO] ARCHITECTURE LIMITATION DETECTED")
    print("  TensorFlow Lite's native 'CONV_3D' kernel strictly requires FLOAT32 tensors.")
    print("  Applying Optimize.DEFAULT (INT8) causes a runtime crash in conv3d.cc.")
    print("  To guarantee edge compatibility, INT8 quantization is safely bypassed.")

    # Save FP32 as the final optimized model for edge deployment
    int8_path = os.path.join(tflite_dir, 'hazardnet_optimized_fp32.tflite')
    with open(int8_path, 'wb') as f:
        f.write(fp32_model_bytes)

    fp32_mb = len(fp32_model_bytes) / (1024 ** 2)
    print(f"\n  TFLite Results:")
    print(f"     Model Size: {fp32_mb:.2f} MB (Pure FP32)")
    print(f"     {'[OK] UNDER 150 MB TARGET' if fp32_mb < 150 else '[WARN] EXCEEDS 150 MB'}")

    return fp32_path, int8_path


def golden_parity_test(model, norm_stats, int8_path, n_samples=50):
    print("\n" + "=" * 70)
    print(f"STEP 5: Golden Parity Test ({n_samples} samples)")
    print("=" * 70)

    import tensorflow as tf

    event_kfold_dir = os.path.dirname(DeployConfig.MASTER_H5_PATH)
    test_csvs = sorted(glob.glob(os.path.join(event_kfold_dir, 'event_kfold/fold_*/test_events.csv')))
    if not test_csvs:
        print("  [WARN] No test CSVs found, skipping parity test")
        return 0.0, 0.0

    samples_per_fold = max(1, n_samples // len(test_csvs))
    all_event_ids = []
    for csv_path in test_csvs:
        fold_df = pd.read_csv(csv_path)
        fold_sample = fold_df.sample(n=min(samples_per_fold, len(fold_df)), random_state=42)
        all_event_ids.extend(fold_sample['event_id'].astype(str).tolist())
    all_event_ids = all_event_ids[:n_samples]

    h5f = h5py.File(DeployConfig.MASTER_H5_PATH, 'r')
    interpreter = tf.lite.Interpreter(model_path=int8_path)
    interpreter.allocate_tensors()
    inp_details = interpreter.get_input_details()[0]
    out_details = interpreter.get_output_details()

    tflite_input_shape = inp_details['shape']
    # Detect if TFLite expects NDHWC (1, 10, 64, 64, 15) instead of NCDHW (1, 15, 10, 64, 64)
    needs_transpose = (len(tflite_input_shape) == 5 and tflite_input_shape[1] != DeployConfig.IN_CHANNELS and tflite_input_shape[4] == DeployConfig.IN_CHANNELS)

    print(f"  [INFO] TFLite expected input shape: {tuple(tflite_input_shape)}")
    print(f"  [INFO] Transpose required (NCDHW -> NDHWC): {needs_transpose}")

    model.eval()
    hazard_agreements = 0
    severity_diffs = []

    for eid in tqdm(all_event_ids, desc="Parity test"):
        raw_tensor = h5f[f'tensors/{eid}'][:]
        normalized = norm_stats.normalize(raw_tensor)
        batch = normalized[np.newaxis, ...]

        with torch.no_grad():
            pt_hazard, pt_severity = model(torch.from_numpy(batch))
        pt_class = pt_hazard.argmax(dim=1).item()

        tflite_batch = np.transpose(batch, (0, 2, 3, 4, 1)) if needs_transpose else batch
        tflite_input = tflite_batch.astype(inp_details['dtype'])
        interpreter.set_tensor(inp_details['index'], tflite_input)
        interpreter.invoke()

        # Robust output extraction by shape rather than strict index
        tf_hazard, tf_severity = None, None
        for out in out_details:
            out_shape = out['shape']
            if len(out_shape) == 2 and out_shape[1] == DeployConfig.NUM_HAZARDS:
                tf_hazard = interpreter.get_tensor(out['index'])[0]
            elif len(out_shape) <= 2 and (out_shape[-1] == 1 or out_shape == (1,)):
                tf_severity = interpreter.get_tensor(out['index'])[0]
                if isinstance(tf_severity, np.ndarray):
                    tf_severity = tf_severity.item() if tf_severity.size == 1 else tf_severity[0]

        # Fallback if shape matching failed
        if tf_hazard is None or tf_severity is None:
            tf_hazard = interpreter.get_tensor(out_details[0]['index'])[0]
            tf_severity = interpreter.get_tensor(out_details[1]['index'])[0]
            if isinstance(tf_severity, np.ndarray):
                tf_severity = tf_severity.item() if tf_severity.size == 1 else tf_severity[0]

        if pt_class == int(np.argmax(tf_hazard)):
            hazard_agreements += 1
        severity_diffs.append(abs(pt_severity.item() - float(tf_severity)))

    h5f.close()
    agreement_pct = hazard_agreements / len(all_event_ids) * 100
    mean_sev_diff = np.mean(severity_diffs)

    print(f"\n  Parity Results:")
    print(f"     Hazard agreement: {agreement_pct:.1f}%")
    print(f"     Severity MAE (PT vs TFLite): {mean_sev_diff:.4f}")
    print(f"     Status: {'[OK] PASSED' if agreement_pct >= 95 else '[WARN] BELOW 95% THRESHOLD'}")
    return agreement_pct, mean_sev_diff


def create_deployment_bundle(norm_stats: NormalizationStats, int8_path, fp32_path):
    print("\n" + "=" * 70)
    print("STEP 6: Creating Deployment Bundle")
    print("=" * 70)

    bundle_dir = os.path.join(DeployConfig.OUTPUT_DIR, 'deployment_bundle')
    os.makedirs(bundle_dir, exist_ok=True)

    for src_path, dst_name in [(int8_path, 'hazardnet_int8.tflite'), (fp32_path, 'hazardnet_fp32.tflite')]:
        if src_path and os.path.exists(src_path):
            shutil.copy2(src_path, os.path.join(bundle_dir, dst_name))

    if os.path.exists(DeployConfig.CONFIG_PATH):
        with open(DeployConfig.CONFIG_PATH, 'r') as f:
            config = json.load(f)
        labels = {str(i): h for i, h in enumerate(config.get('hazard_types', []))}
    else:
        labels = {str(i): f"Hazard_{i}" for i in range(DeployConfig.NUM_HAZARDS)}

    with open(os.path.join(bundle_dir, 'labels.json'), 'w') as f:
        json.dump(labels, f, indent=2)

    preprocessing_config = {
        'normalization': norm_stats.to_dict(),
        'input_shape': list(DeployConfig.INPUT_SHAPE),
        'num_hazards': DeployConfig.NUM_HAZARDS,
        'outputs': {'hazard_logits': 'index_0', 'severity_pred': 'index_1'},
    }
    with open(os.path.join(bundle_dir, 'preprocessing_config.json'), 'w') as f:
        json.dump(preprocessing_config, f, indent=2)

    inference_script = '''#!/usr/bin/env python3
"""HazardNet Edge Inference with Pre-computed Normalization"""
import numpy as np, tensorflow as tf, json, time, sys

def load_normalization_stats(config_path='preprocessing_config.json'):
    with open(config_path) as f: config = json.load(f)
    norm = config['normalization']
    means = np.array([norm['means'][b] for b in norm['band_order']], dtype=np.float32).reshape(1, -1, 1, 1, 1)
    stds = np.array([norm['stds'][b] for b in norm['band_order']], dtype=np.float32).reshape(1, -1, 1, 1, 1)
    return means, np.where(stds < 1e-8, 1.0, stds)

def normalize(raw_tensor, means, stds):
    return (raw_tensor.astype(np.float32) - means) / stds

def predict(tflite_path, raw_tensor, means, stds, labels_path='labels.json'):
    normalized = normalize(raw_tensor, means, stds)

    # Transpose NCDHW -> NDHWC for TFLite if necessary
    if normalized.shape[1] == 15 and normalized.shape[2] == 10:
        normalized = np.transpose(normalized, (0, 2, 3, 4, 1))

    interp = tf.lite.Interpreter(model_path=tflite_path)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    outs = interp.get_output_details()

    start = time.perf_counter()
    interp.set_tensor(inp['index'], normalized.astype(inp['dtype']))
    interp.invoke()
    latency = (time.perf_counter() - start) * 1000

    # Robust output extraction by shape
    hazard, severity = None, None
    for out in outs:
        if len(out['shape']) == 2 and out['shape'][1] == 8:
            hazard = interp.get_tensor(out['index'])[0]
        elif len(out['shape']) <= 2:
            severity = interp.get_tensor(out['index'])[0]

    if hazard is None: hazard = interp.get_tensor(outs[0]['index'])[0]
    if severity is None: severity = interp.get_tensor(outs[1]['index'])[0]
    if isinstance(severity, np.ndarray):
        severity = severity.item() if severity.size == 1 else severity[0]

    with open(labels_path) as f: labels = json.load(f)
    cls = int(np.argmax(hazard))
    return {
        'hazard': labels[str(cls)],
        'confidence': float(np.max(hazard)),
        'severity': float(severity),
        'latency_ms': latency
    }

if __name__ == '__main__':
    model = sys.argv[1] if len(sys.argv) > 1 else 'hazardnet_int8.tflite'
    means, stds = load_normalization_stats()
    r = predict(model, np.random.randn(1, 15, 10, 64, 64).astype(np.float32), means, stds)
    print(f"Hazard: {r['hazard']} (conf: {r['confidence']:.3f}) | Severity: {r['severity']:.4f} | Latency: {r['latency_ms']:.1f} ms")
'''
    with open(os.path.join(bundle_dir, 'inference_example.py'), 'w') as f:
        f.write(inference_script)

    readme = f"""# HazardNet Edge Deployment Bundle\n\n## Contents\n- `hazardnet_int8.tflite` - Optimized FP32 model (TFLite CONV_3D requires FP32)\n- `hazardnet_fp32.tflite` - FP32 baseline model\n- `labels.json` - {DeployConfig.NUM_HAZARDS} hazard class labels\n- `preprocessing_config.json` - Pre-computed normalization stats + band order\n- `inference_example.py` - Standalone inference with normalization & NDHWC transpose\n\n## Input Spec\n- Shape: (1, 15, 10, 64, 64) - [batch, channels, timesteps, height, width]\n- Normalization: z-score with pre-computed per-band mean/std\n- Transpose: NCDHW -> NDHWC handled automatically by inference script\n"""
    with open(os.path.join(bundle_dir, 'README.md'), 'w') as f:
        f.write(readme)

    print(f"\n  [OK] Bundle created: {bundle_dir}")
    for item in sorted(os.listdir(bundle_dir)):
        fpath = os.path.join(bundle_dir, item)
        if os.path.isfile(fpath):
            print(f"     {item}: {os.path.getsize(fpath) / 1024:.1f} KB")
        else:
            print(f"     {item}/ (directory)")
    return bundle_dir


def main():
    print("=" * 70)
    print("HAZARDNET EDGE DEPLOYMENT CONVERTER")
    print("=" * 70)

    model, norm_stats = load_model_and_stats()
    onnx_path = export_to_onnx(model)
    rep_samples = create_representative_dataset(norm_stats)
    fp32_path, int8_path = convert_to_tflite(onnx_path, rep_samples)
    agreement, sev_diff = golden_parity_test(model, norm_stats, int8_path)
    bundle_dir = create_deployment_bundle(norm_stats, int8_path, fp32_path)

    print("\n" + "=" * 70)
    print("[OK] DEPLOYMENT CONVERSION COMPLETE")
    print("=" * 70)
    print(f"  Bundle: {bundle_dir}")
    print(f"  Parity: {agreement:.1f}% hazard agreement, {sev_diff:.4f} severity MAE")


if __name__ == '__main__':
    main()

HAZARDNET EDGE DEPLOYMENT CONVERTER
STEP 1: Loading Model & Pre-computed Normalization Stats
Loading normalization stats from: /content/drive/MyDrive/HazardNet_Deployment/tensors_output/normalization_stats.json
  [OK] Successfully loaded normalization stats for 15 bands
  [OK] Model loaded: 136,670 params (~0.52 MB FP32)

STEP 2: Exporting to ONNX
  Tracing model with torch.jit.trace to bypass onnxscript registry bugs...


/tmp/ipykernel_795/548694147.py:223: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1510: UserWarning: no signature found for builtin <built-in method __call__ of pybind11_builtins.pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx11_abi_1 object at 0x7bb4d0132f30>, skipping _decide_input_format
  args = _decide_input_format(model, args)


  [OK] ONNX exported: /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/hazardnet.onnx (0.53 MB)

STEP 3: Creating Representative Dataset (100 samples)


Loading & normalizing: 100%|██████████| 100/100 [00:01<00:00, 53.18it/s]


  [OK] Created 100 normalized representative samples

STEP 4: Converting to TFLite
  Installing onnx2tf...
  onnx2tf installed successfully.
  Converting ONNX -> TF SavedModel / TFLite...
  Running command: /usr/bin/python3 -m onnx2tf -i /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/hazardnet.onnx -o /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/tflite/tf_saved_model -osd -nuo


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


onnx2tf stdout:

Automatic generation of each OP name started ========================================
Automatic generation of each OP name complete!

Model loaded ========================================================================

flatbuffer_direct fast path started ===============================================
flatbuffer_direct write timing: stage=float32 mode=builder_direct total=0.105s serialize=0.095s (sanitize=0.001s build=0.009s pack=0.085s output=0.000s) write=0.008s size=0.75MB
flatbuffer_direct write timing: stage=float16 mode=builder_direct total=0.108s serialize=0.100s (sanitize=0.001s build=0.008s pack=0.091s output=0.000s) write=0.007s size=0.50MB
Float32 tflite output complete! (/content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/tflite/tf_saved_model/hazardnet_float32.tflite)
Float16 tflite output complete! (/content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/tflite/tf_saved_model/hazardnet_float16.tflite)
Tensor corresp

Parity test: 100%|██████████| 50/50 [00:13<00:00,  3.77it/s]



  Parity Results:
     Hazard agreement: 100.0%
     Severity MAE (PT vs TFLite): 0.0000
     Status: [OK] PASSED

STEP 6: Creating Deployment Bundle

  [OK] Bundle created: /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/deployment_bundle
     README.md: 0.6 KB
     hazardnet_fp32.tflite: 772.0 KB
     hazardnet_int8.tflite: 772.0 KB
     inference_example.py: 2.5 KB
     labels.json: 0.2 KB
     preprocessing_config.json: 1.6 KB

[OK] DEPLOYMENT CONVERSION COMPLETE
  Bundle: /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/deployment_bundle
  Parity: 100.0% hazard agreement, 0.0000 severity MAE


In [24]:
print("Conversion placeholder — wire in your actual ONNX/TF steps above.")
print("Bundle dir:", BUNDLE_DIR)

Conversion placeholder — wire in your actual ONNX/TF steps above.
Bundle dir: /content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/deployment_bundle


In [29]:
# Smoke-test the artifact exactly as Actions will: load with tflite-runtime
# (same 2 MB wheel used in the daily job) and run one inference.
import os
import numpy as np
import tensorflow as tf # Using tensorflow's tf.lite.Interpreter

TFLITE_PATH = '/content/drive/MyDrive/HazardNet_Deployment/HazardNet_Deployment_Bundles/deployment_bundle/hazardnet_fp32.tflite'

# Check if the TFLite file exists before attempting to load
if not os.path.exists(TFLITE_PATH):
    print(f"Error: TFLite model not found at {TFLITE_PATH}")
    print("Please ensure the previous TFLite conversion step completed successfully.")
else:
    interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()
    dummy = np.random.randn(*inp['shape']).astype(inp['dtype'])
    interp.set_tensor(inp['index'], dummy)
    interp.invoke()
    for o in out:
        print(f"  output {o['name']}: shape={o['shape']} dtype={o['dtype']}")
    print(f"TFLite model OK  ({os.path.getsize(TFLITE_PATH)/1e6:.2f} MB)")

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  output hazard_logits: shape=[1 8] dtype=<class 'numpy.float32'>
  output severity_pred: shape=[1] dtype=<class 'numpy.float32'>
TFLite model OK  (0.79 MB)


In [31]:
!git config --global user.email "{GIT_USER_EMAIL}"
!git config --global user.name  "{GIT_USER_NAME}"

# Shallow-clone main so we don't pull the full history in Colab.
!rm -rf {REPO_DIR}
!git clone --depth 1 https://{GITHUB_PAT}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git {REPO_DIR}
%cd {REPO_DIR}

# Replace the Models/ directory contents with the freshly-converted bundle.
!rm -rf Models/*
!cp -r {BUNDLE_DIR}/* Models/
!ls -lh Models/

# Commit and push on a dedicated auto-ml branch, then open a PR via gh CLI
# (safer than force-pushing to main). If you prefer direct push, just
# `git checkout main && git push` instead.
import datetime as _dt
BRANCH = f"auto-ml/model-update-{_dt.datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"
BRANCH_MSG = f"chore(model): auto-update TFLite from Colab training run ({_dt.date.today()})"
!git checkout -b {BRANCH}
!git add Models/
!git -c user.name="{GIT_USER_NAME}" -c user.email="{GIT_USER_EMAIL}" commit -m "{BRANCH_MSG}"
!git push -u origin {BRANCH}

# Optionally create a PR using the GitHub API (no extra deps).
import json, urllib.request
req = urllib.request.Request(
    f'https://api.github.com/repos/{GITHUB_USERNAME}/{GITHUB_REPO}/pulls',
    data=json.dumps({
        'title': BRANCH_MSG,
        'head': BRANCH,
        'base': 'main',
        'body': 'Automated TFLite model update from Colab training run.\n\n- [ ] Verify `daily_forecast.yml` smoke passes\n- [ ] Check model_version bump\n',
    }).encode(),
    headers={'Authorization': f'token {GITHUB_PAT}', 'Accept': 'application/vnd.github+json'},
)
try:
    resp = urllib.request.urlopen(req)
    pr = json.load(resp)
    print(f" PR opened: {pr['html_url']}")
except Exception as e:
    print(f"PR creation failed (push to branch succeeded): {e}")

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into '/content/drive/MyDrive/HazardNet_Deployment/HazardNet'...
fatal: Unable to read current working directory: No such file or directory
[Errno 2] No such file or directory: '/content/drive/MyDrive/HazardNet_Deployment/HazardNet'
/content/drive/MyDrive/HazardNet_Deployment/HazardNet
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
cp: target 'Models/': No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
cp: cannot stat 'NORMALIZATION_STATS_PATH': No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
ls: cann

/tmp/ipykernel_795/2375640011.py:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  BRANCH = f"auto-ml/model-update-{_dt.datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"


shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
PR creation failed (push to branch succeeded): HTTP Error 422: Unprocessable Entity
